In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

N = 2000
timestamps = pd.date_range(start="2025-01-01", periods=N, freq="h")

rows = []

for t in timestamps:

    hour = t.hour

    # normal voltage & current
    voltage = np.random.uniform(210, 240)
    current = np.random.uniform(0.5, 5)

    # normal consumption
    consumption = voltage * current / 1000

    # -----------------------------
    # Inject anomaly (5%)
    # -----------------------------
    if np.random.rand() < 0.05:
        current *= np.random.uniform(2, 4)   # abnormal spike
        consumption = voltage * current / 1000
        anomaly = 1
    else:
        anomaly = 0

    rows.append([
        t,
        hour,
        voltage,
        current,
        round(consumption, 2),
        anomaly
    ])

df = pd.DataFrame(rows, columns=[
    "timestamp",
    "hour",
    "voltage",
    "current",
    "consumption",
    "anomaly"   # for testing only
])

df.to_csv("data/anomaly.csv", index=False)

print("🚨 Anomaly dataset created!")

🚨 Anomaly dataset created!


In [3]:
import pandas as pd
from sklearn.ensemble import IsolationForest
import joblib

df = pd.read_csv("data/anomaly.csv")

features = [
    "hour",
    "voltage",
    "current",
    "consumption"
]

X = df[features]

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(X)

joblib.dump(model, "models/anomaly.pkl")

print("🚨 Anomaly model trained!")

🚨 Anomaly model trained!


In [15]:
import joblib
import pandas as pd

model = joblib.load("models/anomaly.pkl")

# -----------------------------
# INPUT (simulate real case)
# -----------------------------
input_data = {
    "hour": 2,
    "voltage": 230,
    "current": 5,   # 🔥 suspicious high
    "consumption": (230 * 5) / 1000
}

df = pd.DataFrame([input_data])

prediction = model.predict(df)[0]

# -----------------------------
# OUTPUT
# -----------------------------
if prediction == -1:
    print("\n🚨 ANOMALY DETECTED (Possible Theft!)")
else:
    print("\n✅ Normal Usage")


✅ Normal Usage


In [8]:
import joblib
import pandas as pd

model = joblib.load("models/anomaly.pkl")

tests = [
    ("Normal Usage", 2),
    ("Slight Increase", 4),
    ("High Usage", 8),
    ("Extreme Spike", 12)
]

for name, current in tests:

    voltage = 230
    consumption = (voltage * current) / 1000

    df = pd.DataFrame([{
        "hour": 20,
        "voltage": voltage,
        "current": current,
        "consumption": consumption
    }])

    pred = model.predict(df)[0]

    print(f"\n{name}")
    print("Result:", "🚨 Anomaly" if pred == -1 else "✅ Normal")


Normal Usage
Result: ✅ Normal

Slight Increase
Result: ✅ Normal

High Usage
Result: 🚨 Anomaly

Extreme Spike
Result: 🚨 Anomaly
